In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 13.	구별로 학원수 비교 : 서울 대치동이나 목동에 사교육이 발달되었다는 가설을 뒷받침할 수 있는 분석
## ① 서울시 교육(상권업종대분류명 이용) 데이터를 df_academy 변수에 할당하고 확인


In [ ]:
df['상권업종대분류명'].unique()

In [ ]:
df_academy = df[(df['상권업종대분류명']=='교육')&(df['시도명']=='서울특별시')]
df_academy.shape

In [ ]:
df_academy.to_csv(r'C:\ai\downloads\shareData\상가정보\ch13_df_academy.csv', index=False)

In [ ]:
df_academy = pd.read_csv(r'C:\ai\downloads\shareData\상가정보\ch13_df_academy.csv')

## ②	df_academy의 상호명별 빈도수 출력(value_counts() / groupby)

In [ ]:
df_academy['상호명'].value_counts()

## ③ df_academy의 상호명별 빈도수 상위 10개 출력

In [ ]:
df_academy['상호명'].value_counts().head(10)

## ④ df_academy의 시군구명별 빈도수 출력 (학원이 가장 많은 구부터 출력)

In [ ]:
df_academy['시군구명'].value_counts()

In [ ]:
df_academy['시군구명'].value_counts().plot(kind='bar', rot=0)
plt.show()

In [ ]:
sns.countplot(data=df_academy, x='시군구명', order=df_academy['시군구명'].value_counts().index)

## ⑤ 어떤 분류의 학원이 많은지 상위 10개의 분류를 academy_count 변수에 할당 후 출력(상권업종소분류명 컬럼 이용)

In [ ]:
group = df_academy.groupby(['상권업종중분류명','상권업종소분류명'])

for (medium, sub), val in group:
    print(medium, '-', sub)

In [ ]:
academy_count = df_academy['상권업종소분류명'].value_counts().head(10)

In [ ]:
academy_count.to_frame()

## ⑥ 상권업종소분류명별 빈도수가 1000 이상인 데이터를  academy_count_1000 변수에 할당

In [ ]:
academy_counts = df_academy['상권업종소분류명'].value_counts()
academy_count_1000 = academy_counts[academy_counts>1000]

In [ ]:
academy_count_1000

In [ ]:
top1000 = df_academy[df_academy['상권업종소분류명'].isin(academy_count_1000.index)]
top1000.shape

In [ ]:
top1000['시군구명'].value_counts().plot(kind='bar', rot=0)
plt.grid(axis='y')

## ⑦ 시군구명, 상권업종소분류명별 상호명 빈도수를 변수에 할당 및 출력

In [ ]:
academy_group = df_academy.groupby(['시군구명', '상권업종소분류명'])['상호명'].count()

In [ ]:
academy_group

## ⑧ 강남구 데이터 시각화(barplot)

In [ ]:
result = academy_group['강남구'].sort_values(ascending=False)
ax = result.head(10).plot(kind='bar', rot = 45)
labels = ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=12)
plt.ylim([0,2700])
plt.grid(axis='y')
plt.show()

In [ ]:
result = academy_group['강남구'].sort_values(ascending=False).head(10)
ax = sns.barplot(x = result.index, y = result)
plt.xticks(rotation=45)
labels = ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=12)
plt.ylim([0,2700])
plt.grid(axis='y')
plt.show()

## ⑨ “법정동명”이 “대치동”과 “목동”인 데이터에서 상권업종소분류명별 빈도수 출력

In [ ]:
df_academy.loc[df_academy['법정동명']=='대치동', '상권업종소분류명'].value_counts()[:3]

In [ ]:
df_academy.loc[df_academy['법정동명']=='목동', '상권업종소분류명'].value_counts()[:3]

In [ ]:
top3 = df_academy.loc[df_academy['법정동명'].isin(['대치동', '목동']), '상권업종소분류명'].value_counts().head(3)
top2 = top3.drop('요가/필라테스 학원')
top2

In [ ]:
df_academy_sel = df_academy[df_academy['상권업종소분류명'].isin(top2.index)]

In [ ]:
df_academy_sel['법정동명'].value_counts().head(10)

## ⑩상권업종소분류명, 시군구명별 상호명 빈도수를 g 변수에 할당 후 출력

In [ ]:
g = df_academy.groupby(['상권업종소분류명', '시군구명'])['상호명'].count()

In [ ]:
g

## ⑪ g 변수 중 상권업종소분류명 “입시·교과학원” 데이터만 시각화 (pandas-plot.bar, pandas-barh, seaborn-barplot)

In [ ]:
result = g['입시·교과학원'].sort_values(ascending=False)
result

In [ ]:
plt.figure(figsize=(18,3))
result.plot(kind='bar', rot=0)
plt.show()

In [ ]:
plt.figure(figsize=(5,13))
result.plot(kind='barh', rot=0)
plt.show()

In [ ]:
plt.figure(figsize=(18,3))
ax = sns.barplot(x=result.index, y=result)
labels = ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=12)
plt.ylim(0,2700)
plt.show()

# 14. 서울시 데이터의 경도와 위도를 산점도로 시각화
## ① df_academy의 경도와 위도를 시군구명별로 색상을 다르게하여 scatterplot으로 시각화


In [ ]:
plt.figure(figsize=(6,5))

ax=sns.scatterplot(data=df_academy, x='경도', y='위도', hue='시군구명', s=10, alpha=0.5)

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., ncol=2)
plt.show()

https://stackoverflow.com/questions/30490740/move-legend-outside-figure-in-seaborn-tsplot : 범례사용

## ② df_academy의 경도와 위도를 상권업종소분류명별로 색상을 다르게하여 scatterplot으로 시각화

In [ ]:
plt.figure(figsize=(9,7))

ax=sns.scatterplot(data=df_academy, x='경도', y='위도', hue='상권업종소분류명', s=5)

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., ncol=2)
plt.show()

In [ ]:
plt.figure(figsize=(9,7))

ax=sns.scatterplot(data=df_academy[df_academy['상권업종소분류명']=='입시·교과학원'], x='경도', y='위도', s=5)

plt.grid(axis='x')
plt.grid(axis='y')

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')

plt.show()

In [ ]:
plt.figure(figsize=(9,7))

ax=sns.scatterplot(data=df_academy_sel, x='경도', y='위도', hue='상권업종소분류명', s=5, palette='Set2')

plt.grid(axis='x')
plt.grid(axis='y')

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., ncol=2)
plt.show()

## ③ “입시·교과학원” 데이터만, 경도와 위도를 시군구명별로 색상을 다르게하여 scatterplot으로 시각화

In [ ]:
plt.figure(figsize=(9,7))

ax=sns.scatterplot(data=df_academy[df_academy['상권업종소분류명']=='입시·교과학원'], x='경도', y='위도', s=5, hue='시군구명')

plt.grid(axis='x')
plt.grid(axis='y')

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., ncol=2)
plt.show()

## ④ “태권도/무술학원” 데이터만, 경도와 위도를 시군구명별로 색상을 다르게하여 scatterplot으로 시각화

In [ ]:
plt.figure(figsize=(9,7))

ax=sns.scatterplot(data=df_academy[df_academy['상권업종소분류명']=='태권도/무술학원'], x='경도', y='위도', s=5, hue='시군구명')

plt.grid(axis='x')
plt.grid(axis='y')

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., ncol=2)
plt.show()

## ⑤ “입시·교과학원” 데이터와 “태권도/무술학원” 데이터의 경도와 위도를 상권업종소분류명별로 색상을 다르게하여 scatterplot으로 시각화

In [ ]:
plt.figure(figsize=(9,7))

ax=sns.scatterplot(data=df_academy[df_academy['상권업종소분류명'].isin(['입시·교과학원','태권도/무술학원'])], x='경도', y='위도', s=5, hue='상권업종소분류명', palette='hsv')

plt.grid(axis='x')
plt.grid(axis='y')

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., ncol=2)
plt.show()

# 15. 지도시각화 : Folium
` 아나콘다 프롬프트에서 아래의 둘 중 하나를 실행`

`pip install folium`

`conda install -c conda-forge folium`

-	docs : https://python-visualization.github.io/folium/latest/getting_started.html?utm_source=chatgpt.com
-	Quickstart : https://python-visualization.github.io/folium/version-v0.9.1/quickstart.html?utm_source=chatgpt.com


In [ ]:
# import folium
# m = folium.Map(location=(37.4865,126.9295),
#               zoom_start=15,
#               width=800,
#               height=400)
# m

In [ ]:
# 상권업종소분류명이 태권도/무술학원과 입시·교과학원인 데이터
part = ['입시·교과학원','태권도/무술학원']
df_m = df_academy[df_academy['상권업종소분류명'].isin(part)].sample(1000, random_state=3)
df_m.shape

In [ ]:
df_m.head().index

In [ ]:
import folium
lat_mean = df_m['위도'].mean()
long_mean = df_m['경도'].mean()

m = folium.Map(location=(lat_mean,long_mean),
              zoom_start=12,
              width=800,
              height=400)
m

* 지도에 마커 추가하기

In [ ]:
display(df_m.loc[497534, ['상호명','도로명','위도','경도']])
display(df_m.loc[400706, ['상호명','도로명','위도','경도']])

In [ ]:
for i in df_m[:10].index:
    print(i) 

In [ ]:
# m = folium.Map([lat_mean, long_mean], zoom_start=12)

# for i in df_m.index:
#     lat = df_m.loc[i,'위도']
#     long = df_m.loc[i,'경도']
#     tooltip = "<b>{} - {}</b>".format(df_m.loc[i,'상호명'],
#                                      df_m.loc[i,'도로명'][6:])
#     folium.CircleMarker(
#         radius=5,
#         location=[lat, long],
#         tooltip=tooltip,
#         color='#b265e6',
#         fill='#d2b0e8'
#     ).add_to(m)

# m

In [ ]:
# 태권도/무술학원과 입시·교과학원을 분리해서 지도 시각화
df1 = df_m[df_m['상권업종소분류명']=='태권도/무술학원']
df2 = df_m[df_m['상권업종소분류명']=='입시·교과학원']

In [ ]:
# m = folium.Map([lat_mean, long_mean], 
#                zoom_start=12)

# for i in df1.index:
#     lat = df_m.loc[i, '위도']
#     long = df_m.loc[i, '경도']
#     tooltip = '<b>{} - {}</b>'.format(df_m.loc[i, '상호명'],
#                                      df_m.loc[i, '도로명'][6:])
#     folium.Circle(
#         radius=10,
#         location=[lat, long],
#         tooltip=tooltip,
#         color='red',
#         fill=False,
#     ).add_to(m)
    
# for i in df2.index:
#     lat = df_m.loc[i, '위도']
#     long = df_m.loc[i, '경도']
#     tooltip = '<b>{} - {}</b>'.format(df_m.loc[i, '상호명'],
#                                      df_m.loc[i, '도로명'][6:])
#     folium.Circle(
#         radius=10,
#         location=[lat, long],
#         tooltip=tooltip,
#         color='blue',
#         fill=False,
#     ).add_to(m)
# m

In [ ]:
m.save('ch13.html')